## Lightweight OCR Product Extraction - V16 Beam Search + Short False-Hit Guard

V16 starts from the submitted V14 behavior plus the narrow V15 short false-hit guard.

The new larger change is `VIETOCR_BEAMSEARCH=True` for the same VietOCR v1 model. On a paired 80-row train sample, v1 beam search improved the local composite score from 0.5514 to 0.5709 (`+0.0194`). This is still not a guarantee of a 0.67 public score, but it is the strongest evidence-backed change found so far under the relaxed 2-hour runtime cap.

Keep using the VietOCR v1 bundle for this notebook. Do not switch this candidate to v2 unless a new paired validation run proves v2 is better in the full pipeline.


## Cell 1 — Install Dependencies

Run once. Most packages are preinstalled on Kaggle; this cell ensures EasyOCR is available.

In [ ]:
import subprocess
import sys

AUTO_INSTALL_DEPS = True

if AUTO_INSTALL_DEPS:
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "rapidocr",
        "onnxruntime",
        "vietocr",
        "tqdm",
        "scikit-learn",
    ])
    print("Dependencies installed or already available.")
else:
    print("Dependency installation skipped.")


## Cell 2 — Load Data

Read `test.csv`, auto-detect the competition input path, and use `test_images/` (or extract a zip locally).

If `train_labels.csv` is attached to the same Kaggle dataset, it is loaded for optional supervised learning (draft v1 weak labels).

No Internet required — images are read from the competition dataset (see Dataset Description).

In [ ]:
import re, time, zipfile
import pandas as pd
from pathlib import Path

try:
    display
except NameError:
    def display(obj):
        if hasattr(obj, 'to_string'):
            print(obj.to_string(index=False))
        else:
            print(obj)

try:
    from tqdm import tqdm
except ImportError:
    tqdm = lambda iterable, **kwargs: iterable

try:
    from PIL import Image, ImageEnhance, ImageFilter
    HAS_PIL = True
except ImportError:
    HAS_PIL = False


# Kaggle competition mount can be either test_images/images/ or test_images/test_images/images/.
IMAGE_DIR_CANDIDATES = (
    "test_images/test_images/images",
    "test_images/images",
    "test_images",
    "images",
    "test/images",
    "test/test_images/images",
    "test/test_images",
)
IMAGE_ZIP_NAMES = (
    "test_images.zip",
    "images.zip",
    "test/test_images.zip",
    "test/images.zip",
)


def _input_roots() -> list[Path]:
    """Candidate dataset roots: Kaggle competition mount, datasets, local copies."""
    roots: list[Path] = []
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        comp_root = kaggle_input / "competitions"
        if comp_root.exists():
            roots.extend(sorted(comp_root.iterdir()))
        roots.extend(sorted(kaggle_input.iterdir()))
    for local in [Path("data"), Path("../data"), Path("public_dataset"), Path("smce_dataset/test"), Path(".")]:
        if local.exists():
            roots.append(local.resolve())
    # de-dupe while preserving order
    seen: set[Path] = set()
    out: list[Path] = []
    for r in roots:
        if r not in seen:
            seen.add(r)
            out.append(r)
    return out


def resolve_input_dir() -> Path:
    """Auto-detect dataset root containing test.csv (Kaggle competition or uploaded Data)."""
    for root in _input_roots():
        if (root / "test.csv").exists():
            return root
    # nested test.csv (rare uploads)
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        for test_csv in sorted(kaggle_input.rglob("test.csv")):
            return test_csv.parent
    hint = (
        "Dataset not found. On Kaggle: Add Input → Competition Data → "
        "The 2nd URA Hackathon (expect test.csv + test_images/ under "
        "/kaggle/input/competitions/the-2nd-ura-hackathon/)."
    )
    if kaggle_input.exists():
        listing = sorted(str(p) for p in kaggle_input.rglob("*") if p.is_file())[:20]
        hint += f"\n/kaggle/input files (first 20): {listing}"
    raise FileNotFoundError(hint)


def _find_images_dir(input_dir: Path) -> Path | None:
    for rel in IMAGE_DIR_CANDIDATES:
        images_dir = input_dir / rel
        if images_dir.is_dir() and any(images_dir.glob("*.jpg")):
            return images_dir

    # Last-resort Kaggle layout guard: find a jpg under a test_images/test path,
    # but do not accidentally use train_images.
    for jpg in sorted(input_dir.rglob("*.jpg")):
        parts = {part.lower() for part in jpg.parts}
        if "train_images" in parts:
            continue
        if "test_images" in parts or "test" in parts:
            return jpg.parent
    return None


def _find_images_zip(input_dir: Path) -> Path | None:
    for rel in IMAGE_ZIP_NAMES:
        zip_path = input_dir / rel
        if zip_path.is_file():
            return zip_path
    for zip_path in sorted(input_dir.rglob("*.zip")):
        name = zip_path.name.lower()
        if "train" in name and "test" not in name:
            continue
        if name in ("test_images.zip", "images.zip") or name.endswith("images.zip"):
            return zip_path
    return None


def setup_images_dir(input_dir: Path, work_dir: Path) -> Path:
    """Use test_images/ or images/ from input; otherwise extract a zip once to working/."""
    images_dir = _find_images_dir(input_dir)
    if images_dir is not None:
        return images_dir

    zip_path = _find_images_zip(input_dir)
    if zip_path is None:
        raise FileNotFoundError(
            f"No test images under {input_dir}. Expected test_images/, images/, "
            f"or one of {IMAGE_ZIP_NAMES}."
        )

    extract_root = work_dir / "dataset_images"
    images_dir = extract_root / "images"
    if not any(images_dir.glob("*.jpg")):
        extract_root.mkdir(parents=True, exist_ok=True)
        print(f"Extracting {zip_path} ...")
        with zipfile.ZipFile(zip_path, "r") as zf:
            zf.extractall(extract_root)
        # zip may unpack as images/, test_images/, test_images/images/, or nested Kaggle folders.
        if not any(images_dir.glob("*.jpg")):
            for rel in ("test_images/test_images/images", "test_images/images", "test_images", "images"):
                alt = extract_root / rel
                if alt.is_dir() and any(alt.glob("*.jpg")):
                    images_dir = alt
                    break
    return images_dir


INPUT_DIR = resolve_input_dir()
WORK_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
IMAGES_DIR = setup_images_dir(INPUT_DIR, WORK_DIR)

TEST_CSV = INPUT_DIR / "test.csv"
SAMPLE_CSV = INPUT_DIR / "sample_submission.csv"
OUTPUT_CSV = WORK_DIR / "submission.csv"

OCR_BACKEND = "rapidocr_det_vietocr_rec_v16_beam_guard"
CHECKPOINT_CSV = WORK_DIR / f"checkpoint_{OCR_BACKEND}.csv"

# Run controls
RUN_LIMIT = None          # set to 100 for smoke test, None for full Kaggle run
OCR_SCORE_THRESHOLD = 0.35
OCR_MAX_DIM = 1280

# VietOCR model controls
# V12 keeps the v8/v1 recognizer. Edit the first path only if your Kaggle model slug differs.
VIETOCR_WEIGHTS_FILENAME = "vgg_seq2seq_ura.pth"
VIETOCR_CONFIG_FILENAME = "vgg_seq2seq_ura_config.yml"
VIETOCR_BUNDLE_ZIP_FILENAME = "vietocr_vgg_seq2seq_ura_bundle.zip"
VIETOCR_MODEL_INPUT_DIR_CANDIDATES = (
    Path("/kaggle/input/models/solar11781/vietocr-vgg-seq2seq-ura-v1/pytorch/v1/1"),
    Path("/kaggle/input/models/solar11781/vietocr-vgg-seq2seq-ura/pytorch/v1/1"),
    Path("/kaggle/input/vietocr-vgg-seq2seq-ura-bundle-v1"),
    Path("/kaggle/input/vietocr-vgg-seq2seq-ura-v1"),
    Path("vietocr_vgg_seq2seq_ura_bundle-v1"),
)
VIETOCR_MODEL_INPUT_DIR = next(
    (p for p in VIETOCR_MODEL_INPUT_DIR_CANDIDATES if p.exists()),
    VIETOCR_MODEL_INPUT_DIR_CANDIDATES[0],
)
if not VIETOCR_MODEL_INPUT_DIR.exists() and Path("/kaggle/input").exists():
    for _weights_path in sorted(Path("/kaggle/input").rglob(VIETOCR_WEIGHTS_FILENAME)):
        _text = str(_weights_path).lower()
        if "v1" in _text or "bundle-v1" in _text:
            VIETOCR_MODEL_INPUT_DIR = _weights_path.parent
            break
VIETOCR_USE_GPU = False  # falls back to CPU if GPU is unavailable
# V16: beam search costs more recognition time but improved the paired train sample.
# This is intended for the relaxed runtime budget: hard cap is 2 hours on Kaggle CPU.
VIETOCR_BEAMSEARCH = True


# V7 conservative controls. Keep this simple; no train-label memory layers.
CONSERVATIVE_MODE = True
OCR_BLANK_WEAK_TEXT_ONLY = True
OCR_WEAK_TEXT_MAX_CHARS = 24
OCR_WEAK_TEXT_MIN_WORDS = 2
OCR_WEAK_TEXT_MIN_ALPHA_SHARE = 0.35
OCR_WEAK_TEXT_MAX_SYMBOL_SHARE = 0.55

# Product head controls: keep the V12 baseline threshold, then add a guarded V14 recall layer.
PRODUCT_MIN_CLASS_COUNT = 3
PRODUCT_PROB_THRESHOLD = 0.85
PRODUCT_MAX_FEATURES = 3000

# V14 product recall controls.
# Default profile was chosen from a text-only train-label CV sanity check:
#   V12 threshold-only product F1 ~= 0.6418
#   V14 balanced line-refinement product F1 ~= 0.6431
#   V14 aggressive line-refinement product F1 ~= 0.6441
# This is not a hidden-test guarantee. It is only a guard against obviously bad rules.
V14_PRODUCT_RECALL_ENABLED = True
V14_RECALL_PROFILE = "aggressive"  # "off" | "balanced" | "aggressive"
V14_SECOND_PASS_EMPTY_RECALL = True

# V15: very narrow false-hit guard.
# Train probes showed tiny OCR fragments like NAN/Nestle/CANFOCO often came from
# unrelated thumbnails and caused false product predictions. This guard clears
# both OCR and product only for those short product-token hits, while keeping
# source/channel text such as VTV, NEWS, index, Epic, etc.
V15_SHORT_FALSE_PRODUCT_GUARD = True
V15_SHORT_FALSE_PRODUCT_MAX_CHARS = 8
V15_SHORT_FALSE_PRODUCT_KEEP_SOURCE_RE = (
    r"vtv|vtc|vtdc|kenh|tiin|epic|cafef|cafebiz|petro|news|rec|index|"
    r"afamily|sai gon|dan tri|zing|genk|vnexpress|laodong"
)
V15_SHORT_FALSE_PRODUCT_FAMILIES = {
    "nestle",
    "nestle nan",
    "ha long canfoco",
}
V14_SECOND_PASS_PROB_THRESHOLD = 0.80
V14_MAX_PRODUCT_FILL_RATE = 0.54
V14_MAX_CHANGES_VS_V12 = 70
V14_MAX_ADDITIONS_FROM_V12_EMPTY = 30
V14_BLOCKED_SECOND_PASS_PRODUCTS = {
    "VTV",
    "VTV Index",
    "VTVcab SỐNG KHỎE",
}

# Local stratified CV controls. Set RUN_STRATIFIED_CV=True locally, False on Kaggle final.
RUN_STRATIFIED_CV = False
CV_SAMPLE_SIZE = 150
CV_N_SPLITS = 3
CV_RANDOM_STATE = 42

test_df = pd.read_csv(TEST_CSV)
image_ids_on_disk = {p.stem for p in IMAGES_DIR.glob("*.jpg")}

train_labels_df = None
for labels_path in [INPUT_DIR / "train_labels.csv", Path("data/train_labels.csv"), Path("../data/train_labels.csv"), Path("public_dataset/train_labels.csv")]:
    if labels_path.exists():
        train_labels_df = pd.read_csv(labels_path, keep_default_na=False)
        break

print(f"Input dir   : {INPUT_DIR}")
print(f"Test set    : {len(test_df):,} images")
print(f"Images dir  : {IMAGES_DIR} ({len(image_ids_on_disk):,} jpg)")
print(f"Working dir : {WORK_DIR}")
print(f"Conservative: {CONSERVATIVE_MODE} | product_threshold={PRODUCT_PROB_THRESHOLD:.2f} (v14 from v12)")
print(f"VietOCR model dir: {VIETOCR_MODEL_INPUT_DIR}")
print(f"VietOCR beamsearch: {VIETOCR_BEAMSEARCH}  (V16 candidate; keep v1 model bundle)")
print(f"V14 recall  : enabled={V14_PRODUCT_RECALL_ENABLED} | profile={V14_RECALL_PROFILE} | second_pass={V14_SECOND_PASS_EMPTY_RECALL}")
print(f"V15 guard   : short_false_product={V15_SHORT_FALSE_PRODUCT_GUARD} | max_chars={V15_SHORT_FALSE_PRODUCT_MAX_CHARS}")
print(f"Stratified CV: {RUN_STRATIFIED_CV} | sample={CV_SAMPLE_SIZE} | folds={CV_N_SPLITS}")

if train_labels_df is not None:
    ocr_fill = (train_labels_df["ocr_text"].astype(str).str.strip() != "").mean()
    prod_fill = (train_labels_df["product_name"].astype(str).str.strip() != "").mean()
    print(f"Train labels: {len(train_labels_df):,} rows (draft v1 | OCR {ocr_fill:.1%} | product {prod_fill:.1%})")
    print("  Use train.csv + train_images.zip with train_labels_df for supervised training.")
else:
    print("Train labels: not found (test-only mode)")

missing = set(test_df["image_id"]) - image_ids_on_disk
if missing:
    print(f"Warning: {len(missing)} image_ids have no jpg (OCR will return empty for those)")
else:
    print("All test image_ids have matching jpg files")

print("\nPreview test.csv:")
test_df.head(3)


## Cell 3 — Brand Rules (Zero-Parameter Baseline)

Regex dictionary — **no training, no model file**, runs in microseconds.

Use this pattern when you need deterministic, deployable product extraction before adding any ML head.


In [ ]:
# BRAND_RULES: (regex, canonical_name, [product_line_keywords])
BRAND_RULES = [
    # OCR-tolerant Hạ Long / Canfoco / Pate Cột Đèn rules
    (
        r"(ha\s*long|halong|d[o0]\s*h[o0]p\s*ha\s*long|do\s*hop\s*ha\s*long|"
        r"canf[o0]c[o0]|canf[uoa]c[o0]|canfood|canoco|ganfoco)"
        r".{0,80}"
        r"(pate|pa\s*te|pat[eê]|rat[eê]|fat[eê])"
        r".{0,40}"
        r"(cot\s*den|c[o0]t\s*den|c[ộo]t\s*đ[èe]n)",
        "Ha Long Canfoco Pate Cột Đèn",
        []
    ),
    (
        r"(pate|pa\s*te|pat[eê]|rat[eê]|fat[eê])"
        r".{0,40}"
        r"(cot\s*den|c[o0]t\s*den|c[ộo]t\s*đ[èe]n|hai\s*phong|h[ảa]i\s*ph[òo]ng)",
        "Pate Cột Đèn Hải Phòng",
        []
    ),
    (
        r"(cot\s*den|c[o0]t\s*den|c[ộo]t\s*đ[èe]n)"
        r".{0,40}"
        r"(pate|pa\s*te|pat[eê]|rat[eê]|fat[eê])",
        "Pate Cột Đèn Hải Phòng",
        []
    ),
    (
        r"ha\s*long\s*canfoco|halong\s*canfoco|ha\s*long\s*canf[uoa]co|"
        r"canf[o0]co|canf[uoa]co|canfood|canoco|ganfoco",
        "Ha Long Canfoco",
        []
    ),
    (
        r"d[o0]\s*h[o0]p\s*h[aạ]\s*long|do\s*hop\s*ha\s*long|"
        r"cong\s*ty\s*(co\s*phan|cp)\s*do\s*hop\s*ha\s*long",
        "Đồ Hộp Hạ Long",
        []
    ),
    # PATE / HA LONG (dominant in test set)
    (r"ha\s*long\s*canfoco.*pate.*c[ộo]t|c[ộo]t\s*đ[èe]n.*ha\s*long\s*canfoco", "Ha Long Canfoco Pate Cột Đèn", []),
    (r"ha\s*long\s*canfoco.*pate|canfoco.*pate\s*c[ộo]t|pate\s*c[ộo]t\s*đ[èe]n.*canfoco", "Ha Long Canfoco Pate", []),
    (r"ha\s*long\s*canfoco|halong\s*canfoco|canfood|canfoco", "Ha Long Canfoco", []),
    (r"đ[ồo]\s*h[ộo]p\s*h[ạa]\s*long|do\s*hop\s*ha\s*long", "Đồ Hộp Hạ Long", []),
    (r"pate\s*c[ộo]t\s*đ[èe]n|pate\s*cot\s*den|c[ộo]t\s*đ[èe]n\s*h[ảa]i\s*ph[òo]ng", "Pate Cột Đèn Hải Phòng", []),
    (r"h[ạa]\s*long\s*pate|pate\s*h[ạa]\s*long", "Ha Long Canfoco Pate", []),
    # NESTLÉ / NAN / MILO — OCR-tolerant rules
    (
        r"(nestl[eé]|n[eé]stle|nan)"
        r".{0,80}"
        r"(infini\s*pro|infinipro|ifini\s*pro|ifinipro|infi\s*pro|a2)",
        "Nestlé NAN Infinipro A2",
        []
    ),
    (
        r"(nestl[eé]|n[eé]stle|nan)"
        r".{0,80}"
        r"(opti\s*pro|optipro|opti\s*pro\s*plus|optiproplus)",
        "Nestlé NAN Optipro Plus",
        []
    ),
    (
        r"(nestl[eé]|n[eé]stle|nan)"
        r".{0,80}"
        r"(supreme\s*pro|supremepro)",
        "Nestlé NAN Supremepro",
        []
    ),
    (
        r"(nestl[eé]|n[eé]stle|alfamino)"
        r".{0,80}"
        r"(alfamino|infant)",
        "Nestlé Alfamino Infant",
        []
    ),
    (
        r"(nestl[eé]|n[eé]stle|beba)"
        r".{0,80}"
        r"\bbeba\b",
        "Nestlé BEBA",
        []
    ),
    (
        r"\bnan\b|s[uư][aã]\s*nan|sua\s*nan|n[a4]\s*n",
        "Nestlé NAN",
        []
    ),
    (
        r"\bmilo\b|mi1o|miio",
        "Nestlé Milo",
        []
    ),

    # OTHER MILK / INFANT FORMULA
    (
        r"aptamil.{0,60}profutura|profutura.{0,60}aptamil",
        "Aptamil Profutura",
        []
    ),
    (
        r"\baptamil\b",
        "Aptamil",
        []
    ),
    (
        r"\bhipp\b|hi\s*pp|combiotic",
        "HiPP Combiotic",
        []
    ),
    (
        r"optimum\s*gold|0ptimum\s*gold",
        "Optimum Gold",
        []
    ),

    # COFFEE / BEVERAGE BRANDS SEEN IN TRAIN LABELS
    (
        r"highlands?\s*coffee|highlann?ds|higlands?|highlands?",
        "Highlands Coffee",
        []
    ),
    (
        r"the\s*coffee\s*house|coffee\s*house",
        "The Coffee House",
        []
    ),
    (
        r"ph[uú]c\s*long|phuc\s*long",
        "Phúc Long",
        []
    ),

    # SAUCE / OTHER FMCG
    (
        r"chinsu|chin\s*su|t[uư][oơ]ng\s*[oớ]t\s*chinsu|tuong\s*ot\s*chinsu",
        "Chinsu",
        []
    ),
    # MILK / DAIRY
    (r"vinamilk", "Vinamilk", ["flex", "adm gold", "sure", "canxi",
                                 "colosbaby", "colos baby", "ong tho", "ông thọ", "dielac", "grow"]),
    (r"th true|thtrue", "TH True Milk", ["true yogurt", "grow", "school milk", "true butter"]),
    (r"dutch lady|cô gái", "Dutch Lady", ["grow", "complete", "canxi", "mom"]),
    (r"nutifood|nuti\b", "Nutifood", ["growplus", "grow plus", "pedia", "iq"]),
    (r"ensure\b", "Abbott Ensure", ["gold", "original", "plus"]),
    (r"pediasure", "Abbott PediaSure", []),
    (r"similac", "Abbott Similac", []),
    (r"glucerna", "Abbott Glucerna", []),
    (r"milo\b", "Nestlé Milo", []),
    (r"nestle|nestlé", "Nestlé", ["milo", "coffee mate", "carnation", "nestea", "nan", "sữa bột"]),
    (r"aptamil", "Aptamil", []),
    (r"friso\b", "Friso", ["gold", "comfort", "prestige"]),
    (r"meiji\b", "Meiji", ["growing", "step"]),
    (r"ba vi\b|bavi\b|ba vì", "Ba Vì", ["gold"]),
    (r"lothamilk", "Lothamilk", ["canxi"]),
    (r"yomost", "Yomost", []),
    (r"dalat milk|đà lạt", "Đà Lạt Milk", []),
    (r"kun\b|kun milk", "Kun", ["chocolate", "strawberry"]),
    (r"fami\b", "Fami", ["canxi", "kid"]),
    (r"anlene", "Anlene", ["gold", "concentrate"]),
    (r"anchor\b", "Anchor", ["butter", "cream"]),
    # PATE / CANNED MEAT (other brands)
    (r"vissan", "Vissan", ["pate heo", "pate ga", "pate gà",
                           "xuc xich", "xúc xích", "lap xuong", "lạp xưởng"]),
    (r"hafi\b", "Hafi", ["pate"]),
    (r"ba huan|ba huân", "Ba Huân", ["pate"]),
    (r"san ha\b|san hà", "San Hà", ["pate"]),
    (
        r"\bc\.p\.|cp\s*foods|cp\s*food|cp\s*vi[eệ]t\s*nam|c\.p\s*vi[eệ]t\s*nam",
        "CP",
        ["pate", "xúc xích", "xuc xich"]
    ),
    (r"long bien|long biên", "Long Biên", ["pate"]),
    # (r"\bpate\b|patê", "Pate", []),
    # ADD YOUR OWN BELOW
    # (r"regex", "Brand", ["line1", "line2"]),
]

import unicodedata

def fold_text(s: str) -> str:
    """Lowercase + remove Vietnamese accents for OCR-tolerant matching."""
    s = "" if s is None else str(s).lower()
    s = s.replace("đ", "d")
    return "".join(
        c for c in unicodedata.normalize("NFD", s)
        if unicodedata.category(c) != "Mn"
    )

LINE_CANONICAL = {
    "flex": "Flex",
    "adm gold": "ADM Gold",
    "sure": "Sure",
    "canxi": "Canxi",
    "colosbaby": "ColosBaby",
    "colos baby": "ColosBaby",
    "ong tho": "Ông Thọ",
    "ông thọ": "Ông Thọ",
    "dielac": "Dielac",
    "grow": "Grow",
    "grow+": "Grow+",
    "growplus": "Grow Plus",
    "grow plus": "Grow Plus",
    "optipro": "Optipro",
    "opti pro": "Optipro",
    "infinipro": "Infinipro",
    "infini pro": "Infinipro",
    "pate heo": "Pate Heo",
    "pate ga": "Pate Gà",
    "pate gà": "Pate Gà",
    "pate gan": "Pate Gan",
    "xuc xich": "Xúc Xích",
    "xúc xích": "Xúc Xích",
    "lap xuong": "Lạp Xưởng",
    "lạp xưởng": "Lạp Xưởng",
}

def extract_product(text: str) -> str:
    """Return 'Brand ProductLine', 'Brand', or '' if no match."""
    if not text or not text.strip():
        return ""
    raw = text.lower().replace("patê", "pate")
    tl = raw + " " + fold_text(raw)
    for pattern, brand, lines in BRAND_RULES:
        if re.search(pattern, tl, re.IGNORECASE):
            for line in lines:
                if re.search(line, tl, re.IGNORECASE):
                    line_key = line.lower().strip()
                    line_canon = LINE_CANONICAL.get(line_key, line.strip().title())
                    return f"{brand} {line_canon}"
            return brand
    return ""

tests = [
    ("HA LONG CANFOCO Pate Cột Đèn tạm dừng sản xuất", "Ha Long Canfoco Pate Cột Đèn"),
    ("HALONG CANFOCO pate cot den Hải Phòng", "Ha Long Canfoco Pate Cột Đèn"),
    ("ĐỒ HỘP HẠ LONG ISO 22000", "Đồ Hộp Hạ Long"),
    ("Pate cột đèn ai khóc nỗi đau này", "Pate Cột Đèn Hải Phòng"),
    ("Sữa tươi Vinamilk Flex 180ml giảm 20%", "Vinamilk Flex"),
    ("MILO Nestle chocolate 3 in 1", "Nestlé Milo"),
    ("Pate Heo Vissan 170g combo 3 hộp", "Vissan Pate Heo"),
    ("TH True Milk tươi tiệt trùng ít béo", "TH True Milk"),
    ("Dutch Lady Grow+ 900g", "Dutch Lady Grow"),
    ("No brand in this text", ""),
]
all_pass = True
for text, expected in tests:
    got = extract_product(text)
    ok = got == expected
    all_pass = all_pass and ok
    status = "PASS" if ok else "FAIL"
    print(f"{status}: '{text[:45]}' -> got='{got}' | expected='{expected}'")

print()
print("All self-tests passed." if all_pass else "Some self-tests failed — check BRAND_RULES.")
print(f"Total rules loaded: {len(BRAND_RULES)}")


In [ ]:
EXTRA_BRAND_RULES = [
    # Strong Hạ Long / Canfoco variants from OCR noise.
    (
        r"(halong|ha\s*long|ha\s*lonc|h[ạa]\s*long|h\s*long|hl)"
        r".{0,80}"
        r"(canfoco|cafoco|cafo?co|canfuco|canfood|canoco|ganfoco|halor\s*canfoco)",
        "Ha Long Canfoco",
        []
    ),
    (
        r"(cong\s*ty|c[oô]ng\s*ty|cty)"
        r".{0,40}"
        r"(cp|c[oô]\s*phan|cổ\s*phần)"
        r".{0,60}"
        r"(d[o0ổ]\s*h[oộeệ]p|do\s*hop)"
        r".{0,30}"
        r"(h[ạa]\s*long|ha\s*long|h\s*long)",
        "Đồ Hộp Hạ Long",
        []
    ),
    (
        r"(d[o0ổ]\s*h[oộeệ]p|do\s*hop|đồ\s*hộp|đổ\s*hộp)"
        r".{0,25}"
        r"(h[ạa]\s*long|ha\s*long|h\s*long|hl)",
        "Đồ Hộp Hạ Long",
        []
    ),

    # Pate Cột Đèn variants: OCR often turns Patê into Ratê/Fatê/Latê/Batê/Eatê/Zatê.
    (
        r"(pate|pat[eê]|pa\s*te|rat[eê]|fat[eê]|lat[eê]|bat[eê]|eat[eê]|zat[eê])"
        r".{0,50}"
        r"(c[o0]t\s*den|c[ộo]t\s*đ[èe]n|hai\s*phong|h[ảa]i\s*ph[òo]ng)",
        "Pate Cột Đèn Hải Phòng",
        []
    ),
    (
        r"(c[o0]t\s*den|c[ộo]t\s*đ[èe]n)"
        r".{0,50}"
        r"(pate|pat[eê]|pa\s*te|rat[eê]|fat[eê]|lat[eê]|bat[eê]|eat[eê]|zat[eê])",
        "Pate Cột Đèn Hải Phòng",
        []
    ),

    # Safe direct brand/product rules.
    (r"hoa\s*sen\s*home", "HOA SEN HOME", []),
    (r"highlands?\s*coffee|highland\s*coffee|higlands?\s*coffee", "Highlands Coffee", []),
    (r"coffee\s*house|the\s*coffee\s*house", "The Coffee House", []),
    (r"ph[uú]c\s*long|phuc\s*long", "Phúc Long", []),
]

# Prepend so these stronger rules run before the older generic rules.
BRAND_RULES = EXTRA_BRAND_RULES + BRAND_RULES

print(f"Extra rules added: {len(EXTRA_BRAND_RULES)}")
print(f"Total rules now : {len(BRAND_RULES)}")

## Cell 3b — Train Lightweight Product Head

Trains in **seconds on CPU**. Serialized size typically **2–5 MB**.

| Component | Size | Inference |
|---|---|---|
| `BRAND_RULES` | 0 MB | ~0.01 ms |
| Sklearn head (this cell) | ~few MB | ~0.5 ms |
| EasyOCR (Cell 4) | ~200 MB | ~1–2 s/img |

**Pattern:** rules first → binary "has product?" gate → TF-IDF char n-grams + LogisticRegression.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline


def canonicalize_product_name(product_name: str, ocr_text: str = "") -> str:
    """Clean noisy product labels after rules/sklearn prediction."""
    name = "" if product_name is None else str(product_name).strip()
    if not name:
        return ""

    folded = fold_text(ocr_text)
    name_folded = fold_text(name)

    # Hạ Long / Canfoco family
    has_pate = re.search(r"pate|pa\s*te|rat[eê]|fat[eê]", folded)
    has_cot_den = re.search(r"cot\s*den|hai\s*phong", folded)
    has_canfoco = re.search(
        r"ha\s*long\s*canfoco|halong\s*canfoco|canfoco|canfuco|canfood|canoco|ganfoco",
        folded
    )
    has_do_hop_ha_long = re.search(
        r"do\s*hop\s*ha\s*long|cong\s*ty\s*(co\s*phan|cp)\s*do\s*hop\s*ha\s*long",
        folded
    )
    
    if has_canfoco and has_pate and has_cot_den:
        return "Ha Long Canfoco Pate Cột Đèn"
    
    if has_pate and has_cot_den:
        return "Pate Cột Đèn Hải Phòng"
    
    if has_canfoco:
        return "Ha Long Canfoco"
    
    if has_do_hop_ha_long:
        return "Đồ Hộp Hạ Long"

    # Nestlé / NAN family
    if re.search(r"\bnan\b", folded):
        if re.search(r"opti\s*pro|optipro", folded):
            return "Nestlé NAN Optipro Plus"
        if re.search(r"infini\s*pro|infinipro|infipro", folded):
            return "Nestlé NAN Infinipro A2"
        if re.search(r"alfamino", folded):
            return "Nestlé Alfamino Infant"
        return "Nestlé NAN"

    if re.search(r"milo", folded):
        return "Nestlé Milo"

    if re.search(r"nestle|nestlé", folded):
        return "Nestlé"

    # Avoid generic false positive
    if folded in {"pate", "pa te", "patê"}:
        return ""

    # Normalize exact label variants only
    replacements = {
        "halong canfoco": "Ha Long Canfoco",
        "halong cafoco": "Ha Long Canfoco",
        "ha long canfoco": "Ha Long Canfoco",
        "do hop ha long halong canfoco": "Ha Long Canfoco",
        "cong ty co phan do hop ha long": "Đồ Hộp Hạ Long",
        "do hop cong ty cp do hop ha long": "Đồ Hộp Hạ Long",
        "do hop ha long": "Đồ Hộp Hạ Long",
        "pate cot den": "Pate Cột Đèn Hải Phòng",
        "pate cot den hai phong": "Pate Cột Đèn Hải Phòng",
        "pate ha long": "Pate Hạ Long",
        "pate cot den ha long canfoco": "Ha Long Canfoco Pate Cột Đèn",
        "nan": "Nestlé NAN",
        "sua nan": "Nestlé NAN",
        "sua nan": "Nestlé NAN",
    }

    return replacements.get(name_folded, name)

def normalize_for_evidence(s: str) -> str:
    """Fold accents and normalize common OCR confusions for evidence matching."""
    s = fold_text("" if s is None else str(s))

    # OCR often reads Pate/Patê as Ratê/Fatê/Latê/Pajê in this dataset.
    s = re.sub(
        r"\b(rat[eê]?|fat[eê]?|lat[eê]?|bat[eê]?|eat[eê]?|zat[eê]?|pat[eê]?|paj[eê]?)\b",
        "pate",
        s
    )

    # OCR variants seen around Cột Đèn / Hạ Long / Canfoco
    s = s.replace("canfuco", "canfoco")
    s = s.replace("canfood", "canfoco")
    s = s.replace("canoco", "canfoco")
    s = s.replace("ganfoco", "canfoco")
    s = s.replace("cafoco", "canfoco")
    s = s.replace("halor canfoco", "ha long canfoco")
    s = s.replace("halong", "ha long")
    s = s.replace("ha lonc", "ha long")
    s = re.sub(r"\bh\s*long\b", "ha long", s)
    s = re.sub(r"\bdo\s*hep\b|\bdo\s*hop\b|\bdo\s*hop\b", "do hop", s)
    
    # Nestlé / NAN / infant formula OCR variants
    s = s.replace("neste", "nestle")
    s = s.replace("nestlé", "nestle")
    s = s.replace("optiproplus", "optipro plus")
    s = s.replace("opti proplus", "optipro plus")
    s = re.sub(r"\bopti\s*pro\b", "optipro", s)
    s = re.sub(r"\binfini\s*pro\b|\binfi\s*pro\b|\bifini\s*pro\b|\bifinipro\b", "infinipro", s)
    s = re.sub(r"\bn\s*a\s*n\b", "nan", s)

    # Coffee brand OCR variants
    s = s.replace("cofee", "coffee")
    s = s.replace("coffe", "coffee")
    s = re.sub(r"\bhighlann?ds\b|\bhiglands\b|\bhighland\b", "highlands", s)

    # Other OCR variants
    s = re.sub(r"\bhi\s*pp\b", "hipp", s)
    s = re.sub(r"\bchin\s*su\b", "chinsu", s)
    s = re.sub(r"\bphuc\s*long\b", "phuc long", s)

    s = re.sub(r"[^a-z0-9]+", " ", s)
    return re.sub(r"\s+", " ", s).strip()


GENERIC_PRODUCT_NAMES = {
    "pate",
    "pa te",
    "thit heo",
    "thit hop",
    "san pham do hop",
    "do hop",
    "sua",
    "sua bot",
}


PRODUCT_STOP_TOKENS = {
    "nestle", "ha", "long", "do", "hop", "cong", "ty", "co", "phan",
    "hai", "phong", "sua", "pate", "patê", "milk", "food", "foods",
}


def product_supported_by_ocr(product_name: str, ocr_text: str) -> bool:
    """
    Return True only if the predicted product has visible OCR evidence.
    This prevents the sklearn head from hallucinating learned labels.
    """
    name_norm = normalize_for_evidence(product_name)
    ocr_norm = normalize_for_evidence(ocr_text)

    if not name_norm or not ocr_norm:
        return False

    if name_norm in GENERIC_PRODUCT_NAMES:
        return False

    # Exact normalized phrase visible in OCR.
    if name_norm in ocr_norm:
        return True

    name_tokens = name_norm.split()
    ocr_tokens = set(ocr_norm.split())

    # Product-specific canonical expansions are allowed only if the distinctive line is visible.
    if "nan" in name_tokens:
        return "nan" in ocr_tokens

    if "milo" in name_tokens:
        return "milo" in ocr_tokens

    if "cot" in name_tokens and "den" in name_tokens:
        return (
            "pate" in ocr_tokens
            and (
                {"cot", "den"}.issubset(ocr_tokens)
                or {"hai", "phong"}.issubset(ocr_tokens)
            )
        )

    if "canfoco" in name_tokens:
        return (
            "canfoco" in ocr_tokens
            or {"ha", "long"}.issubset(ocr_tokens)
            or {"do", "hop", "ha", "long"}.issubset(ocr_tokens)
        )

    if name_norm == "do hop ha long":
        return {"do", "hop", "ha", "long"}.issubset(ocr_tokens)

    if name_norm == "cp":
        if {"do", "hop", "ha", "long"}.issubset(ocr_tokens):
            return False

        return (
            "cp foods" in ocr_norm
            or "cp food" in ocr_norm
            or "cp vietnam" in ocr_norm
            or "cp viet nam" in ocr_norm
            or "c p vietnam" in ocr_norm
            or "c p viet nam" in ocr_norm
        )

    if name_norm == "ba vi":
        return {"ba", "vi"}.issubset(ocr_tokens) or "bavi" in ocr_tokens

    if name_norm == "phuc long":
        return {"phuc", "long"}.issubset(ocr_tokens)

    if name_norm == "the coffee house":
        return "coffee" in ocr_tokens and "house" in ocr_tokens

    if name_norm == "highlands coffee":
        return "highlands" in ocr_tokens

    if name_norm == "nhan hoa foods":
        return {"nhan", "hoa"}.issubset(ocr_tokens)

    if name_norm == "hipp combiotic":
        return "hipp" in ocr_tokens or "combiotic" in ocr_tokens

    if name_norm.startswith("aptamil"):
        return "aptamil" in ocr_tokens

    if name_norm == "optimum gold":
        return {"optimum", "gold"}.issubset(ocr_tokens)

    # Generic fallback: require at least one distinctive non-generic token.
    distinctive = [
        t for t in name_tokens
        if len(t) >= 3 and t not in PRODUCT_STOP_TOKENS
    ]

    if not distinctive:
        return False

    return any(t in ocr_tokens for t in distinctive)


def safe_product(product_name: str, ocr_text: str) -> str:
    """
    Canonicalize, then reject the product if OCR does not support it.
    """
    name = canonicalize_product_name(product_name, ocr_text)
    if not name:
        return ""

    if product_supported_by_ocr(name, ocr_text):
        return name

    return ""


def v14_refine_detected_product(product_name: str, ocr_text: str) -> tuple[str, str]:
    """
    Strengthen a product label only when V12 already detected the family and
    OCR contains explicit product-line evidence. This avoids broad test-set
    blending and keeps changes explainable.
    """
    if not V14_PRODUCT_RECALL_ENABLED or V14_RECALL_PROFILE == "off":
        return product_name, ""

    product_name = "" if product_name is None else str(product_name).strip()
    if not product_name:
        return "", ""

    evidence = normalize_for_evidence(ocr_text)
    tokens = set(evidence.split())
    product_norm = normalize_for_evidence(product_name).replace("coffeee", "coffee")
    profile = str(V14_RECALL_PROFILE).lower().strip()

    if product_norm in {"nestle", "nestle nan", "nestle nan optipro plus"} or product_norm.startswith("nestle"):
        if "nan" in tokens and ("infinipro" in tokens or "a2" in tokens):
            return "Nestlé NAN Infinipro A2", "v14_refine_nan_infinipro"
        if "nan" in tokens and ("optipro" in tokens or "plus" in tokens):
            return "Nestlé NAN Optipro Plus", "v14_refine_nan_optipro"
        if "nan" in tokens and ("supremepro" in tokens or "supreme" in tokens):
            return "Nestlé NAN Supremepro", "v14_refine_nan_supreme"
        if "alfamino" in tokens and ("infant" in tokens or "formula" in tokens):
            return "Nestlé Alfamino Infant", "v14_refine_alfamino"
        if "beba" in tokens:
            return "Nestlé BEBA", "v14_refine_beba"

    if product_norm == "aptamil":
        if "profutura" in tokens:
            return "Aptamil Profutura", "v14_refine_aptamil_profutura"
        if {"infant", "formula"}.issubset(tokens) or {"from", "birth"}.issubset(tokens):
            return "Aptamil Infant Formula", "v14_refine_aptamil_infant"

    # This line-refinement helped the train-label CV slightly, but it is still
    # profile-gated because some labels keep this family generic.
    if profile == "aggressive" and product_norm == "highlands coffee":
        if "tra" in tokens and "sen" in tokens and "vang" in tokens:
            return "Highlands Coffee Trà Sen Vàng", "v14_refine_highlands_tra_sen"

    return product_name, ""


def v14_candidate_allowed(product_name: str, ocr_text: str) -> bool:
    """Final guard for lower-threshold second-pass product additions."""
    product_name = "" if product_name is None else str(product_name).strip()
    if not product_name:
        return False
    if product_name in V14_BLOCKED_SECOND_PASS_PRODUCTS:
        return False
    if _short_v14_ocr_evidence(product_name, ocr_text):
        return False
    return product_supported_by_ocr(product_name, ocr_text)


def _short_v14_ocr_evidence(product_name: str, ocr_text: str) -> bool:
    evidence = normalize_for_evidence(ocr_text)
    name_norm = normalize_for_evidence(product_name)
    if not evidence:
        return True
    words = evidence.split()
    if name_norm == "nestle nan" and (evidence in {"nan", "viet nan"} or len(words) <= 2):
        return True
    return len(evidence) <= 20 or len(words) <= 2


v14_second_pass_product_predictor = None

class ProductPredictor:
        def __init__(self, min_class_count=3, prob_threshold=0.60, max_features=3000):
            self.min_class_count = min_class_count
            self.prob_threshold = prob_threshold
            self.max_features = max_features
            self._has_clf = self._prod_clf = None
            self._n_train = self._n_classes = 0

        def fit(self, train_labels, rule_fn):
            df = train_labels.copy()
            df["ocr_text"] = df["ocr_text"].astype(str).str.strip()
            df["product_name"] = [
                canonicalize_product_name(p, o)
                for p, o in zip(df["product_name"], df["ocr_text"])
            ]
            self._rule_fn = rule_fn
            self._has_clf = Pipeline([
                ("tfidf", TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), max_features=self.max_features, min_df=2)),
                ("clf", LogisticRegression(max_iter=400, class_weight="balanced")),
            ])
            self._has_clf.fit(df["ocr_text"], (df["product_name"] != "").astype(int))
            pos = df[(df["ocr_text"] != "") & (df["product_name"] != "")]
            keep = pos["product_name"].value_counts()
            keep = keep[keep >= self.min_class_count].index
            pos = pos[pos["product_name"].isin(keep)]
            self._prod_clf = Pipeline([
                ("tfidf", TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), max_features=self.max_features, min_df=2)),
                ("clf", LogisticRegression(max_iter=400, class_weight="balanced")),
            ])
            if len(pos):
                self._prod_clf.fit(pos["ocr_text"], pos["product_name"])
            self._n_train, self._n_classes = len(df), pos["product_name"].nunique() if len(pos) else 0
            return self

        def summary(self):
            return (
                f"ProductPredictor("
                f"train_rows={self._n_train}, "
                f"classes={self._n_classes}, "
                f"threshold={self.prob_threshold}, "
                f"max_features={self.max_features})"
            )

        def model_size_mb(self):
            import pickle
            try:
                size_bytes = len(pickle.dumps(self))
                return size_bytes / (1024 * 1024)
            except Exception:
                return float("nan")

        def predict_with_meta(self, ocr_text):
            ocr_text = "" if ocr_text is None else str(ocr_text).strip()
            meta = {"source": "empty", "has_product_prob": 0.0, "raw_candidate": ""}
            if not ocr_text:
                return "", meta

            # 1) Rules are safest because they are directly triggered by OCR text.
            ruled = self._rule_fn(ocr_text)
            if ruled:
                safe = safe_product(ruled, ocr_text)
                if safe:
                    meta.update(source="rule", raw_candidate=ruled, has_product_prob=1.0)
                    return safe, meta

            # 2) Classifier is allowed only as a fallback, and only if OCR supports the label.
            if self._has_clf is None or self._prod_clf is None or 1 not in self._has_clf.classes_:
                meta["source"] = "no_model"
                return "", meta

            proba = self._has_clf.predict_proba([ocr_text])[0]
            has_product_prob = float(proba[list(self._has_clf.classes_).index(1)])
            meta["has_product_prob"] = has_product_prob
            if has_product_prob < self.prob_threshold:
                meta["source"] = "below_threshold"
                return "", meta

            pred = str(self._prod_clf.predict([ocr_text])[0])
            safe = safe_product(pred, ocr_text)
            if safe:
                meta.update(source="classifier", raw_candidate=pred)
                return safe, meta

            meta.update(source="classifier_rejected", raw_candidate=pred)
            return "", meta

        def predict(self, ocr_text):
            return self.predict_with_meta(ocr_text)[0]


product_predictor = None
if train_labels_df is not None:
    product_predictor = ProductPredictor(
        min_class_count=PRODUCT_MIN_CLASS_COUNT,
        prob_threshold=PRODUCT_PROB_THRESHOLD,
        max_features=PRODUCT_MAX_FEATURES,
    )
    product_predictor.fit(train_labels_df, extract_product)

    if V14_PRODUCT_RECALL_ENABLED and V14_SECOND_PASS_EMPTY_RECALL:
        v14_second_pass_product_predictor = ProductPredictor(
            min_class_count=PRODUCT_MIN_CLASS_COUNT,
            prob_threshold=V14_SECOND_PASS_PROB_THRESHOLD,
            max_features=PRODUCT_MAX_FEATURES,
        )
        v14_second_pass_product_predictor.fit(train_labels_df, extract_product)

    pos = train_labels_df.copy()
    pos["ocr_text"] = pos["ocr_text"].astype(str).str.strip()
    pos["product_name"] = pos["product_name"].astype(str).str.strip()
    n_pairs = ((pos["ocr_text"] != "") & (pos["product_name"] != "")).sum()
    print(f"Trained lightweight product head on {len(train_labels_df):,} rows ({n_pairs:,} OCR+product pairs)")
    print(product_predictor.summary())
    if v14_second_pass_product_predictor is not None:
        print("  V14 second pass:", v14_second_pass_product_predictor.summary())
    print(f"  Product head size: ~{product_predictor.model_size_mb():.2f} MB")

    import time
    t0 = time.perf_counter()
    for _ in range(500):
        product_predictor.predict("Vinamilk Flex 180ml Ha Long Canfoco")
    ms = (time.perf_counter() - t0) / 500 * 1000
    print(f"  Product inference: ~{ms:.2f} ms/image (CPU, after OCR)")
else:
    print("train_labels.csv not found — rules-only mode (simplest lightweight path)")


def predict_product_with_meta(ocr_text: str) -> tuple[str, dict]:
    if product_predictor is not None:
        product_name, meta = product_predictor.predict_with_meta(ocr_text)
    else:
        product_name = safe_product(extract_product(ocr_text), ocr_text)
        meta = {"source": "rule" if product_name else "empty", "has_product_prob": 1.0 if product_name else 0.0, "raw_candidate": product_name}

    if V14_PRODUCT_RECALL_ENABLED:
        if not product_name and v14_second_pass_product_predictor is not None:
            second_product, second_meta = v14_second_pass_product_predictor.predict_with_meta(ocr_text)
            if v14_candidate_allowed(second_product, ocr_text):
                product_name = second_product
                meta = {
                    **second_meta,
                    "source": f"v14_second_pass_{second_meta.get('source', 'unknown')}",
                    "raw_candidate": second_meta.get("raw_candidate", second_product),
                }

        refined_name, refine_source = v14_refine_detected_product(product_name, ocr_text)
        if refine_source and refined_name != product_name and product_supported_by_ocr(refined_name, ocr_text):
            meta = {
                **meta,
                "source": refine_source,
                "raw_candidate": product_name,
            }
            product_name = refined_name

    return product_name, meta


def predict_product(ocr_text: str) -> str:
    return predict_product_with_meta(ocr_text)[0]


In [ ]:

from sklearn.model_selection import StratifiedKFold
import numpy as np


def product_token_f1(gt, pred):
    gt = "" if pd.isna(gt) else str(gt).strip()
    pred = "" if pd.isna(pred) else str(pred).strip()

    if not gt and not pred:
        return 1.0

    gt_tokens = set(gt.lower().split())
    pred_tokens = set(pred.lower().split())

    if not gt_tokens or not pred_tokens:
        return 0.0

    common = gt_tokens & pred_tokens
    if not common:
        return 0.0

    precision = len(common) / len(pred_tokens)
    recall = len(common) / len(gt_tokens)

    return 2 * precision * recall / (precision + recall)


def cer_score(gt: str, pred: str) -> float:
    gt = "" if pd.isna(gt) else str(gt).strip()
    pred = "" if pd.isna(pred) else str(pred).strip()
    if not gt:
        return 1.0 if not pred else 0.0

    previous = list(range(len(pred) + 1))
    for i, gt_char in enumerate(gt, start=1):
        current = [i]
        for j, pred_char in enumerate(pred, start=1):
            current.append(
                previous[j - 1]
                if gt_char == pred_char
                else 1 + min(previous[j - 1], previous[j], current[j - 1])
            )
        previous = current
    return 1.0 - min(previous[-1] / len(gt), 1.0)


def score_prediction_frame(solution: pd.DataFrame, prediction: pd.DataFrame) -> dict:
    merged = solution[["image_id", "ocr_text", "product_name"]].merge(
        prediction[["image_id", "ocr_text", "product_name"]],
        on="image_id",
        suffixes=("_gt", "_pred"),
        how="inner",
    )
    product_scores = [
        product_token_f1(gt, pred)
        for gt, pred in zip(merged["product_name_gt"], merged["product_name_pred"])
    ]
    ocr_scores = [
        cer_score(gt, pred)
        for gt, pred in zip(merged["ocr_text_gt"], merged["ocr_text_pred"])
    ]
    product_f1 = float(np.mean(product_scores))
    ocr_score = float(np.mean(ocr_scores))
    return {
        "score": float(0.6 * product_f1 + 0.4 * ocr_score),
        "product_f1": product_f1,
        "ocr_score": ocr_score,
        "rows": int(len(merged)),
        "product_fill": float((prediction["product_name"].astype(str).str.strip() != "").mean()),
        "ocr_fill": float((prediction["ocr_text"].astype(str).str.strip() != "").mean()),
    }


def product_family_label(row: pd.Series) -> str:
    has_ocr = str(row["ocr_text"]).strip() != ""
    product = normalize_for_evidence(row["product_name"])
    if not product:
        return f"empty_product|ocr={int(has_ocr)}"
    families = [
        ("ha_long", r"ha\s+long|canfoco|do\s+hop|cot\s+den|pate"),
        ("nan", r"\bnan\b|nestle"),
        ("milo", r"\bmilo\b"),
        ("coffee", r"highlands|coffee|phuc\s+long"),
        ("milk", r"vinamilk|dutch\s+lady|true\s+milk|aptamil|ensure|pediasure|friso|meiji|hipp"),
    ]
    for family, pattern in families:
        if re.search(pattern, product):
            return f"{family}|ocr={int(has_ocr)}"
    return f"other_product|ocr={int(has_ocr)}"


def make_cv_strata(df: pd.DataFrame, n_splits: int) -> pd.Series:
    strata = df.apply(product_family_label, axis=1)
    counts = strata.value_counts()
    rare = set(counts[counts < n_splits].index)
    if rare:
        fallback = [
            f"product={int(str(product).strip() != '')}|ocr={int(str(ocr).strip() != '')}"
            for product, ocr in zip(df["product_name"], df["ocr_text"])
        ]
        strata = pd.Series(
            [fallback_value if value in rare else value for value, fallback_value in zip(strata, fallback)],
            index=df.index,
        )
    return strata


def find_train_images_dir() -> Path | None:
    for path in [
        INPUT_DIR / "train_images/train_images",
        INPUT_DIR / "train_images",
        Path("data/train_images/train_images"),
        Path("../data/train_images/train_images"),
    ]:
        if path.exists() and any(path.glob("*.jpg")):
            return path
    return None


def evaluate_product_predictor(model, val_df):
    val_pred = val_df["ocr_text"].map(model.predict)
    scores = [
        product_token_f1(gt, pred)
        for gt, pred in zip(val_df["product_name"], val_pred)
    ]
    return {
        "product_f1": float(np.mean(scores)),
        "fill_rate": float((val_pred.astype(str).str.strip() != "").mean()),
    }


def run_stratified_product_cv(labels: pd.DataFrame, n_splits: int = CV_N_SPLITS) -> pd.DataFrame:
    clean = labels[["image_id", "ocr_text", "product_name"]].copy().reset_index(drop=True)
    strata = make_cv_strata(clean, n_splits)
    splitter = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=CV_RANDOM_STATE)
    rows = []
    for fold, (train_idx, val_idx) in enumerate(splitter.split(clean, strata), start=1):
        train_fold = clean.iloc[train_idx].reset_index(drop=True)
        val_fold = clean.iloc[val_idx].reset_index(drop=True)
        model = ProductPredictor(
            min_class_count=PRODUCT_MIN_CLASS_COUNT,
            prob_threshold=PRODUCT_PROB_THRESHOLD,
            max_features=PRODUCT_MAX_FEATURES,
        ).fit(train_fold, extract_product)
        metrics = evaluate_product_predictor(model, val_fold)
        rows.append({"fold": fold, **metrics, "rows": len(val_fold)})
    return pd.DataFrame(rows)


if train_labels_df is not None:
    product_cv = run_stratified_product_cv(train_labels_df)
    print("Stratified product CV using ground-truth OCR:")
    display(product_cv)
    print("Mean:")
    display(product_cv.mean(numeric_only=True).to_frame("mean").T)
else:
    print("train_labels.csv not found — skip product-head CV.")


## Cell 4 ? OCR Engine (RapidOCR Detection + VietOCR Recognition)

RapidOCR is used only to detect text boxes. The text for every accepted box is recognized with your trained VietOCR model.

Model files must be available through `VIETOCR_MODEL_INPUT_DIR` in the config cell.


In [ ]:
def patch_numpy_private_api_for_vietocr() -> None:
    """Patch NumPy 2.x removals expected by VietOCR/imgaug."""
    import numpy as _np

    if not hasattr(_np, "sctypes"):
        _np.sctypes = {
            "int": [_np.int8, _np.int16, _np.int32, _np.int64],
            "uint": [_np.uint8, _np.uint16, _np.uint32, _np.uint64],
            "float": [_np.float16, _np.float32, _np.float64],
            "complex": [_np.complex64, _np.complex128],
            "others": [_np.bool_, _np.object_, _np.bytes_, _np.str_],
        }

    for name, value in {
        "bool": _np.bool_,
        "int": int,
        "float": float,
        "complex": complex,
        "object": object,
    }.items():
        if name not in _np.__dict__:
            setattr(_np, name, value)

    if not hasattr(_np, "_codex_original_fromstring"):
        _np._codex_original_fromstring = _np.fromstring

        def fromstring_compat(string, dtype=float, count=-1, sep="", *, like=None):
            if isinstance(string, (bytes, bytearray, memoryview)) and (sep is None or sep == ""):
                return _np.frombuffer(string, dtype=dtype, count=count)
            if like is None:
                return _np._codex_original_fromstring(string, dtype=dtype, count=count, sep=sep)
            return _np._codex_original_fromstring(string, dtype=dtype, count=count, sep=sep, like=like)

        _np.fromstring = fromstring_compat


def patch_pillow_private_api_for_rapidocr() -> None:
    """Patch Pillow private helper expected by RapidOCR on newer Kaggle images."""
    try:
        import os
        import PIL._util as pil_util
    except Exception:
        return

    if hasattr(pil_util, "is_directory"):
        return

    def is_directory(path) -> bool:
        try:
            return os.path.isdir(path)
        except TypeError:
            return False

    pil_util.is_directory = is_directory


def resolve_vietocr_model_paths() -> tuple[Path, Path]:
    """Return explicit VietOCR weights/config paths, or extract the configured bundle zip."""
    model_dir = Path(VIETOCR_MODEL_INPUT_DIR)

    search_dirs = [model_dir]
    if model_dir.exists():
        search_dirs.extend([p for p in sorted(model_dir.rglob("*")) if p.is_dir() and p != model_dir])

    for root in search_dirs:
        weights_path = root / VIETOCR_WEIGHTS_FILENAME
        config_path = root / VIETOCR_CONFIG_FILENAME
        if weights_path.exists() and config_path.exists():
            return weights_path, config_path

    zip_candidates = []
    if model_dir.exists():
        direct_zip = model_dir / VIETOCR_BUNDLE_ZIP_FILENAME
        if direct_zip.exists():
            zip_candidates.append(direct_zip)
        zip_candidates.extend(
            p for p in sorted(model_dir.rglob("*.zip"))
            if p not in zip_candidates and ("v1" in str(p).lower() or p.name == VIETOCR_BUNDLE_ZIP_FILENAME)
        )

    for zip_path in zip_candidates:
        extract_dir = WORK_DIR / "vietocr_model_from_bundle_v1"
        extract_dir.mkdir(parents=True, exist_ok=True)
        if not ((extract_dir / VIETOCR_WEIGHTS_FILENAME).exists() and (extract_dir / VIETOCR_CONFIG_FILENAME).exists()):
            print(f"Extracting VietOCR model bundle: {zip_path}")
            with zipfile.ZipFile(zip_path, "r") as zf:
                zf.extractall(extract_dir)
        weights_path = extract_dir / VIETOCR_WEIGHTS_FILENAME
        config_path = extract_dir / VIETOCR_CONFIG_FILENAME
        if weights_path.exists() and config_path.exists():
            return weights_path, config_path

    raise FileNotFoundError(
        "VietOCR v1 model files were not found. On Kaggle, add/upload a dataset containing either:\n"
        f"  {model_dir / VIETOCR_WEIGHTS_FILENAME}\n"
        f"  {model_dir / VIETOCR_CONFIG_FILENAME}\n"
        "or this bundle zip at the dataset root:\n"
        f"  {model_dir / VIETOCR_BUNDLE_ZIP_FILENAME}\n"
        "If your Kaggle dataset slug is different, edit VIETOCR_MODEL_INPUT_DIR_CANDIDATES in the config cell."
    )

patch_numpy_private_api_for_vietocr()
patch_pillow_private_api_for_rapidocr()

# If a prior same-kernel run failed mid-import, clear partial RapidOCR modules.
import sys as _sys
for _mod_name in list(_sys.modules):
    if _mod_name == "rapidocr" or _mod_name.startswith("rapidocr."):
        del _sys.modules[_mod_name]

try:
    from rapidocr import RapidOCR
    RAPIDOCR_API = "rapidocr"
except ImportError as rapidocr_error:
    try:
        from rapidocr_onnxruntime import RapidOCR
        RAPIDOCR_API = "rapidocr_onnxruntime"
    except ImportError as fallback_error:
        raise ImportError(
            "Could not import RapidOCR after applying the Pillow compatibility patch. "
            "Original rapidocr error: " + repr(rapidocr_error) +
            "; fallback rapidocr_onnxruntime error: " + repr(fallback_error)
        ) from rapidocr_error

rapidocr_engine = RapidOCR()
print(f"RapidOCR detector loaded ({RAPIDOCR_API})")

try:
    import torch
except ImportError as torch_error:
    raise ImportError(
        "PyTorch is required for VietOCR recognition. Kaggle usually has torch preinstalled; "
        "for local runs, install torch inside this project's .venv before running the OCR cells."
    ) from torch_error
from vietocr.tool.config import Cfg
from vietocr.tool.predictor import Predictor

VIETOCR_WEIGHTS_PATH, VIETOCR_CONFIG_PATH = resolve_vietocr_model_paths()
VIETOCR_DEVICE = "cuda:0" if VIETOCR_USE_GPU and torch.cuda.is_available() else "cpu"

vietocr_config = Cfg.load_config_from_file(str(VIETOCR_CONFIG_PATH))
vietocr_config["weights"] = str(VIETOCR_WEIGHTS_PATH)
vietocr_config["device"] = VIETOCR_DEVICE
if "cnn" in vietocr_config:
    vietocr_config["cnn"]["pretrained"] = False
if "predictor" in vietocr_config:
    vietocr_config["predictor"]["beamsearch"] = bool(VIETOCR_BEAMSEARCH)

vietocr_predictor = Predictor(vietocr_config)
print(f"VietOCR recognizer loaded on {VIETOCR_DEVICE}")
print(f"  weights: {VIETOCR_WEIGHTS_PATH}")
print(f"  config : {VIETOCR_CONFIG_PATH}")


def load_image(image_id: str):
    """Load image from images/."""
    path = IMAGES_DIR / f"{image_id}.jpg"
    if not path.exists():
        return None
    try:
        return Image.open(path).convert("RGB")
    except Exception:
        return None


def preprocess(img, max_dim: int = OCR_MAX_DIM):
    """Resize, sharpen lightly, and boost contrast for OCR."""
    w, h = img.size
    if max(w, h) > max_dim:
        ratio = max_dim / max(w, h)
        img = img.resize((int(w * ratio), int(h * ratio)), Image.LANCZOS)
    img = ImageEnhance.Contrast(img).enhance(1.35)
    return img.filter(ImageFilter.SHARPEN)


def postprocess_ocr(text: str) -> str:
    """Normalize whitespace and drop consecutive duplicate tokens."""
    text = re.sub(r"\s+", " ", text).strip()
    tokens = text.split()
    if not tokens:
        return ""

    deduped = [tokens[0]]
    for tok in tokens[1:]:
        if tok.lower() != deduped[-1].lower():
            deduped.append(tok)

    return " ".join(deduped)


def _rapidocr_sort_key(box):
    """Sort OCR boxes roughly top-to-bottom, left-to-right."""
    arr = np.asarray(box)
    if arr.ndim == 2 and arr.shape[1] >= 2:
        return float(arr[:, 1].min()), float(arr[:, 0].min())
    return 0.0, 0.0


def unpack_rapidocr_result(result) -> list[tuple]:
    """Return [(box, rapid_text, score), ...] for rapidocr and rapidocr_onnxruntime."""
    txts = getattr(result, "txts", None)
    if txts is not None:
        txts = list(txts)
        boxes = getattr(result, "boxes", None)
        scores = getattr(result, "scores", None)
        scores = list(scores) if scores is not None else [1.0] * len(txts)
        if boxes is not None:
            return list(zip(list(boxes), txts, scores))
        return [(None, text, score) for text, score in zip(txts, scores)]

    if isinstance(result, tuple):
        result = result[0]
    if not result:
        return []
    items = []
    for row in result:
        if len(row) >= 3:
            items.append((row[0], row[1], float(row[2])))
    return items


def crop_text_box(img: Image.Image, box, pad_ratio: float = 0.06) -> Image.Image:
    """Crop a RapidOCR text box for VietOCR recognition."""
    if box is None:
        return img.convert("RGB")

    arr = np.asarray(box, dtype=np.float32)
    if arr.ndim != 2 or arr.shape[1] < 2:
        return img.convert("RGB")

    x1, y1 = arr[:, 0].min(), arr[:, 1].min()
    x2, y2 = arr[:, 0].max(), arr[:, 1].max()
    pad = max(2, int(max(x2 - x1, y2 - y1) * pad_ratio))
    crop = img.crop((
        max(0, int(x1) - pad),
        max(0, int(y1) - pad),
        min(img.width, int(x2) + pad),
        min(img.height, int(y2) + pad),
    ))
    return crop.convert("RGB")


def recognize_vietocr_lines(img: Image.Image, items: list[tuple]) -> list[str]:
    """Recognize every accepted RapidOCR-detected crop with VietOCR."""
    lines = []
    for box, _rapid_text, score in items:
        if score < OCR_SCORE_THRESHOLD:
            continue
        crop = crop_text_box(img, box)
        try:
            pred = str(vietocr_predictor.predict(crop)).strip()
        except Exception as exc:
            print(f"VietOCR crop recognition failed: {repr(exc)}")
            pred = ""
        if pred:
            lines.append(pred)
    return lines


def weak_ocr_text_only(ocr_text: str, product_name: str) -> bool:
    """Blank text-only OCR that is likely too weak/noisy to help scoring."""
    if not CONSERVATIVE_MODE or not OCR_BLANK_WEAK_TEXT_ONLY or product_name:
        return False
    text = "" if ocr_text is None else str(ocr_text).strip()
    if not text:
        return False
    visible = [ch for ch in text if not ch.isspace()]
    if not visible:
        return True
    words = re.findall(r"\w+", text, flags=re.UNICODE)
    alpha_share = sum(ch.isalpha() for ch in visible) / max(len(visible), 1)
    symbol_share = sum(not ch.isalnum() for ch in visible) / max(len(visible), 1)
    return (
        len(text) <= OCR_WEAK_TEXT_MAX_CHARS
        or len(words) < OCR_WEAK_TEXT_MIN_WORDS
        or alpha_share < OCR_WEAK_TEXT_MIN_ALPHA_SHARE
        or symbol_share > OCR_WEAK_TEXT_MAX_SYMBOL_SHARE
    )


def v15_short_false_product_hit(ocr_text: str, product_name: str) -> bool:
    """Clear tiny product-token false hits found by train probes and visual audit."""
    if not V15_SHORT_FALSE_PRODUCT_GUARD:
        return False
    text = "" if ocr_text is None else str(ocr_text).strip()
    product_name = "" if product_name is None else str(product_name).strip()
    if not text or not product_name or len(text) > V15_SHORT_FALSE_PRODUCT_MAX_CHARS:
        return False

    evidence = normalize_for_evidence(text)
    product_norm = normalize_for_evidence(product_name)
    if not evidence or product_norm not in V15_SHORT_FALSE_PRODUCT_FAMILIES:
        return False
    if re.search(V15_SHORT_FALSE_PRODUCT_KEEP_SOURCE_RE, evidence, flags=re.IGNORECASE):
        return False

    return evidence in product_norm or evidence in {"nan", "nestle", "canfoco"}


def run_ocr(image_id: str) -> tuple:
    """Return (ocr_text, product_name), using RapidOCR boxes + VietOCR text recognition."""
    img = load_image(image_id)
    if img is None:
        return "", ""

    img = preprocess(img)
    img_np = np.array(img)

    try:
        if RAPIDOCR_API == "rapidocr":
            result = rapidocr_engine(img_np, text_score=OCR_SCORE_THRESHOLD)
        else:
            result = rapidocr_engine(img_np)

        items = unpack_rapidocr_result(result)
        if not items:
            ocr_text = ""
        else:
            items = sorted(items, key=lambda x: _rapidocr_sort_key(x[0]) if x[0] is not None else (0.0, 0.0))
            lines = recognize_vietocr_lines(img, items)
            ocr_text = postprocess_ocr(" ".join(lines))

    except Exception as e:
        print(f"RapidOCR detection / VietOCR recognition failed on {image_id}: {repr(e)}")
        ocr_text = ""
    product_name, product_meta = predict_product_with_meta(ocr_text)
    if v15_short_false_product_hit(ocr_text, product_name):
        ocr_text = ""
        product_name = ""
    elif weak_ocr_text_only(ocr_text, product_name):
        ocr_text = ""
    return ocr_text, product_name


print("\nSmoke test on first image...")
first_id = test_df["image_id"].iloc[0]
ocr, prod = run_ocr(first_id)
prod_rules = extract_product(ocr)

print(f"  image_id    : {first_id}")
print(f"  ocr_text    : {ocr[:80]}{'...' if len(ocr) > 80 else ''}")
print(f"  product     : {prod or '(empty)'}")

if prod != prod_rules:
    print(f"  rules-only  : {prod_rules or '(empty)'}  (train model override)")


## Cell 4b — Stratified Local CV

Runs full-pipeline CV on a stratified sample of train images. Keep `RUN_STRATIFIED_CV=False` on Kaggle final submissions.


In [ ]:

def make_cv_sample(labels: pd.DataFrame, sample_size: int, seed: int, n_splits: int) -> pd.DataFrame:
    labels = labels[["image_id", "ocr_text", "product_name"]].copy().reset_index(drop=True)
    if sample_size is None or sample_size <= 0 or sample_size >= len(labels):
        return labels

    strata = make_cv_strata(labels, n_splits)
    sampled_parts = []
    for _, idx in labels.groupby(strata).groups.items():
        part = labels.loc[list(idx)]
        n = max(1, round(sample_size * len(part) / len(labels)))
        sampled_parts.append(part.sample(min(n, len(part)), random_state=seed))
    sampled = pd.concat(sampled_parts).sample(frac=1, random_state=seed).head(sample_size)
    return sampled.reset_index(drop=True)


def run_stratified_pipeline_cv(labels: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    train_images_dir = find_train_images_dir()
    if train_images_dir is None:
        raise FileNotFoundError("train_images folder not found; expected data/train_images/train_images locally.")

    cv_labels = make_cv_sample(labels, CV_SAMPLE_SIZE, CV_RANDOM_STATE, CV_N_SPLITS)
    strata = make_cv_strata(cv_labels, CV_N_SPLITS)
    splitter = StratifiedKFold(n_splits=CV_N_SPLITS, shuffle=True, random_state=CV_RANDOM_STATE)

    global IMAGES_DIR, product_predictor
    old_images_dir = IMAGES_DIR
    old_predictor = product_predictor

    rows = []
    fold_metrics = []
    try:
        IMAGES_DIR = train_images_dir
        for fold, (train_idx, val_idx) in enumerate(splitter.split(cv_labels, strata), start=1):
            train_fold = cv_labels.iloc[train_idx].reset_index(drop=True)
            val_fold = cv_labels.iloc[val_idx].reset_index(drop=True)
            product_predictor = ProductPredictor(
                min_class_count=PRODUCT_MIN_CLASS_COUNT,
                prob_threshold=PRODUCT_PROB_THRESHOLD,
                max_features=PRODUCT_MAX_FEATURES,
            ).fit(train_fold, extract_product)

            fold_rows = []
            print(f"Running fold {fold}/{CV_N_SPLITS}: {len(val_fold)} images")
            for image_id in tqdm(list(val_fold["image_id"]), desc=f"CV fold {fold}"):
                ocr_text, product_name = run_ocr(image_id)
                fold_rows.append({
                    "fold": fold,
                    "image_id": image_id,
                    "ocr_text": ocr_text,
                    "product_name": product_name,
                })

            pred_fold = pd.DataFrame(fold_rows)
            metrics = score_prediction_frame(val_fold, pred_fold)
            metrics["fold"] = fold
            fold_metrics.append(metrics)
            rows.extend(fold_rows)
    finally:
        IMAGES_DIR = old_images_dir
        product_predictor = old_predictor

    pred_df = pd.DataFrame(rows)
    cv_metrics = pd.DataFrame(fold_metrics)
    return pred_df, cv_metrics


cv_predictions_df = None
cv_metrics_df = None

if RUN_STRATIFIED_CV and train_labels_df is not None:
    t_cv = time.perf_counter()
    cv_predictions_df, cv_metrics_df = run_stratified_pipeline_cv(train_labels_df)
    cv_elapsed = time.perf_counter() - t_cv

    print("\nStratified full-pipeline CV metrics:")
    display(cv_metrics_df)
    mean_metrics = cv_metrics_df[["score", "product_f1", "ocr_score", "product_fill", "ocr_fill"]].mean()
    print(
        f"CV mean score={mean_metrics['score']:.6f} | "
        f"product_f1={mean_metrics['product_f1']:.6f} | "
        f"ocr_score={mean_metrics['ocr_score']:.6f} | "
        f"product_fill={mean_metrics['product_fill']:.1%} | "
        f"ocr_fill={mean_metrics['ocr_fill']:.1%}"
    )
    print(f"CV runtime: {cv_elapsed/60:.1f} min for {len(cv_predictions_df):,} images")
else:
    print("Stratified full-pipeline CV skipped. Set RUN_STRATIFIED_CV=True locally to run it.")


## Cell 5 — Main Loop

Processes test images on **CPU**. Most time is OCR (~1–2 s/image); product head adds <1 ms.

Expected composite score: **~0.5** (reference starter — improve accuracy while keeping models small for Final Presentation).

In [ ]:
SAVE_EVERY = 50

done_ids = set()
results = []

if CHECKPOINT_CSV.exists():
    ckpt = pd.read_csv(CHECKPOINT_CSV, keep_default_na=False)
    done_ids = set(ckpt["image_id"])
    results = ckpt.to_dict("records")
    print(f"Resuming from checkpoint: {len(done_ids):,} images done")
else:
    print("Starting fresh")

source_ids = test_df["image_id"].head(RUN_LIMIT) if RUN_LIMIT else test_df["image_id"]
pending = [i for i in source_ids if i not in done_ids]
print(f"Pending: {len(pending):,} | Done: {len(done_ids):,}")
print()

empty_ocr_outputs = 0
t_start = time.perf_counter()

for idx, img_id in enumerate(tqdm(pending, desc="OCR Progress")):
    ocr_text, product_name = run_ocr(img_id)

    if ocr_text == "" and (IMAGES_DIR / f"{img_id}.jpg").exists():
        empty_ocr_outputs += 1

    results.append({
        "image_id": img_id,
        "ocr_text": ocr_text,
        "product_name": product_name,
    })

    if (idx + 1) % SAVE_EVERY == 0:
        pd.DataFrame(results).to_csv(CHECKPOINT_CSV, index=False, encoding="utf-8")

pd.DataFrame(results).to_csv(CHECKPOINT_CSV, index=False, encoding="utf-8")

df_result = pd.DataFrame(results)
ocr_fill = (df_result["ocr_text"].str.strip() != "").mean()
prod_fill = (df_result["product_name"].str.strip() != "").mean()

print()
print(f"Processed     : {len(df_result):,}")
print(f"OCR fill rate : {ocr_fill:.1%}")
print(f"Product fill  : {prod_fill:.1%}")
elapsed = time.perf_counter() - t_start
sec_per_img = elapsed / max(len(pending), 1)

print(f"Empty OCR outs: {empty_ocr_outputs:,}  (some test images have no readable text)")
print(f"Runtime       : {elapsed/60:.1f} min | {sec_per_img:.2f} sec/image")

## Cell 6 — Validate and Export submission.csv

Validate submission format before uploading to Kaggle. All checks must pass.

In [ ]:
import csv
import sys

sys.path.insert(0, str(Path(".").resolve()))
try:
    from build_dataset import write_submission_csv
except ImportError:
    def write_submission_csv(df, path):
        out = df[["image_id", "ocr_text", "product_name"]].copy()
        for col in ("ocr_text", "product_name"):
            out[col] = out[col].fillna("").astype(str).str.strip()
            out.loc[out[col] == "", col] = " "
        out.to_csv(path, index=False, encoding="utf-8", quoting=csv.QUOTE_ALL)

sub = pd.DataFrame(results)[["image_id", "ocr_text", "product_name"]]
sample = pd.read_csv(SAMPLE_CSV, keep_default_na=False)

print("Validating submission format...\n")
checks = {}

expected_ids = set(sample["image_id"])
got_ids = set(sub["image_id"])
checks["AC-1 Row count match"] = len(sub) == len(sample)
checks["AC-2 No extra IDs"] = len(got_ids - expected_ids) == 0
checks["AC-3 No missing IDs"] = len(expected_ids - got_ids) == 0
checks["AC-4 No duplicate IDs"] = not sub["image_id"].duplicated().any()
checks["AC-5 No null values"] = not sub.isnull().any().any()
checks["AC-6 No newline in text"] = not sub["ocr_text"].str.contains(r"\n|\t", regex=True, na=False).any()
checks["AC-7 Columns correct"] = list(sub.columns) == ["image_id", "ocr_text", "product_name"]

all_pass = True
for name, ok in checks.items():
    status = "PASS" if ok else "FAIL"
    print(f"  [{status}] {name}")
    if not ok:
        all_pass = False

print()
if all_pass:
    sub = sub.set_index("image_id").reindex(sample["image_id"]).reset_index()
    sub["ocr_text"] = sub["ocr_text"].fillna("").astype(str).str.strip()
    sub["product_name"] = sub["product_name"].fillna("").astype(str).str.strip()
    write_submission_csv(sub, OUTPUT_CSV)

    print("All checks passed.")
    print(f"Saved to: {OUTPUT_CSV}")
    print("\nFirst 5 rows:")
    display(sub.head())
    ocr_fill = (sub["ocr_text"].str.strip() != "").mean()
    prod_fill = (sub["product_name"].str.strip() != "").mean()
    print(f"\nStats: OCR fill={ocr_fill:.1%} | Product fill={prod_fill:.1%} | Rows={len(sub):,}")
    print("\nNext steps:")
    print("  1. Download submission.csv from /kaggle/working/")
    print("  2. Compare it locally against v8/v9 before submitting")
    print("  3. Submit only if the private-safe gates pass")
else:
    print("Validation failed — fix issues above before submitting.")


In [ ]:
def _clean_submission_frame(df: pd.DataFrame) -> pd.DataFrame:
    out = df[["image_id", "ocr_text", "product_name"]].copy()
    for col in out.columns:
        out[col] = out[col].fillna("").astype(str).str.strip()
    return out


def _load_reference_submission(candidates: list[Path], label: str) -> pd.DataFrame | None:
    for path in candidates:
        if path.exists():
            ref = _clean_submission_frame(pd.read_csv(path, keep_default_na=False))
            print(f"Loaded {label} reference: {path} ({len(ref):,} rows)")
            return ref
    print(f"{label} reference not found; skipping reference-based risk gates.")
    return None


def _short_product_evidence(product_name: str, ocr_text: str) -> bool:
    evidence = normalize_for_evidence(ocr_text)
    name_norm = normalize_for_evidence(product_name)
    if not product_name or not evidence:
        return False
    words = evidence.split()
    if name_norm == "nestle nan" and (evidence in {"nan", "viet nan"} or len(words) <= 2):
        return True
    return len(evidence) <= 20 or len(words) <= 2


def _unsupported_product_rows(df: pd.DataFrame) -> pd.DataFrame:
    audit = _clean_submission_frame(df)
    non_empty = audit[audit["product_name"] != ""].copy()
    non_empty["supported_by_ocr"] = [
        product_supported_by_ocr(p, o)
        for p, o in zip(non_empty["product_name"], non_empty["ocr_text"])
    ]
    return non_empty[~non_empty["supported_by_ocr"]].copy()


def audit_product_evidence(df):
    non_empty = _clean_submission_frame(df)
    non_empty = non_empty[non_empty["product_name"] != ""].copy()
    bad = _unsupported_product_rows(df)

    print(f"Non-empty product predictions: {len(non_empty):,}")
    print(f"Unsupported by OCR evidence   : {len(bad):,}")

    if len(bad):
        display(bad.head(30))

    return bad


def private_safe_risk_audit(candidate: pd.DataFrame) -> pd.DataFrame:
    candidate = _clean_submission_frame(candidate)
    baseline_v8 = _load_reference_submission(
        [
            Path("data/submission_v8.csv"),
            Path("../data/submission_v8.csv"),
            Path("/kaggle/input/submission-v8/submission_v8.csv"),
            Path("/kaggle/input/private-safe-baseline/submission_v8.csv"),
        ],
        "v8 baseline",
    )
    baseline_v12 = _load_reference_submission(
        [
            Path("data/submission_v12.csv"),
            Path("../data/submission_v12.csv"),
            Path("/kaggle/input/submission-v12/submission_v12.csv"),
            Path("/kaggle/input/private-safe-baseline/submission_v12.csv"),
        ],
        "v12 baseline",
    )
    prior_v9 = _load_reference_submission(
        [
            Path("data/submission_v9.csv"),
            Path("../data/submission_v9.csv"),
            Path("/kaggle/input/submission-v9/submission_v9.csv"),
        ],
        "v9 comparison",
    )

    ocr_fill = float((candidate["ocr_text"] != "").mean())
    product_fill = float((candidate["product_name"] != "").mean())
    unsupported = audit_product_evidence(candidate)

    rows = [
        {"gate": "format_rows", "value": len(candidate), "limit": len(test_df), "pass": len(candidate) == len(test_df)},
        {"gate": "product_fill_rate", "value": product_fill, "limit": V14_MAX_PRODUCT_FILL_RATE, "pass": product_fill <= V14_MAX_PRODUCT_FILL_RATE},
        {"gate": "ocr_fill_rate", "value": ocr_fill, "limit": None, "pass": True},
        {"gate": "unsupported_products_report_only", "value": len(unsupported), "limit": None, "pass": True},
    ]

    if baseline_v8 is not None:
        merged = baseline_v8.merge(candidate, on="image_id", suffixes=("_v8", "_cand"))
        additions = merged[(merged["product_name_v8"] == "") & (merged["product_name_cand"] != "")].copy()
        changes = merged[merged["product_name_v8"] != merged["product_name_cand"]].copy()
        short_additions = additions[
            [
                _short_product_evidence(p, o)
                for p, o in zip(additions["product_name_cand"], additions["ocr_text_cand"])
            ]
        ].copy()

        v8_unsupported = _unsupported_product_rows(baseline_v8)
        v8_ocr_fill = float((baseline_v8["ocr_text"] != "").mean())

        rows.extend([
            {"gate": "product_additions_from_v8_empty", "value": len(additions), "limit": 50, "pass": len(additions) <= 50},
            {"gate": "short_ocr_product_additions", "value": len(short_additions), "limit": 0, "pass": len(short_additions) == 0},
            {"gate": "product_changes_vs_v8", "value": len(changes), "limit": 180, "pass": len(changes) <= 180},
            {"gate": "unsupported_products_vs_v8", "value": len(unsupported), "limit": len(v8_unsupported), "pass": len(unsupported) <= len(v8_unsupported)},
            {"gate": "ocr_fill_not_below_v8_minus_1pct", "value": ocr_fill, "limit": v8_ocr_fill - 0.01, "pass": ocr_fill >= v8_ocr_fill - 0.01},
        ])

        print("\nPrivate-safe drift vs v8:")
        print(f"  Product additions from v8 empty: {len(additions):,}")
        print(f"  Short-OCR product additions    : {len(short_additions):,}")
        print(f"  Product changes vs v8          : {len(changes):,}")
        print(f"  Product fill candidate/v8      : {product_fill:.2%} / {(baseline_v8['product_name'] != '').mean():.2%}")
        print(f"  OCR fill candidate/v8          : {ocr_fill:.2%} / {v8_ocr_fill:.2%}")

        if len(short_additions):
            print("\nShort-OCR additions to inspect before any submission:")
            display(short_additions[["image_id", "product_name_cand", "ocr_text_cand"]].head(30))

        if len(additions):
            print("\nProduct additions from v8-empty rows:")
            display(additions[["image_id", "product_name_cand", "ocr_text_cand"]].head(30))


    if baseline_v12 is not None:
        merged12 = baseline_v12.merge(candidate, on="image_id", suffixes=("_v12", "_cand"))
        additions12 = merged12[(merged12["product_name_v12"] == "") & (merged12["product_name_cand"] != "")].copy()
        removals12 = merged12[(merged12["product_name_v12"] != "") & (merged12["product_name_cand"] == "")].copy()
        swaps12 = merged12[
            (merged12["product_name_v12"] != "")
            & (merged12["product_name_cand"] != "")
            & (merged12["product_name_v12"] != merged12["product_name_cand"])
        ].copy()
        changes12 = merged12[merged12["product_name_v12"] != merged12["product_name_cand"]].copy()
        short_additions12 = additions12[
            [
                _short_product_evidence(p, o)
                for p, o in zip(additions12["product_name_cand"], additions12["ocr_text_cand"])
            ]
        ].copy()
        v12_unsupported = _unsupported_product_rows(baseline_v12)
        v12_ocr_fill = float((baseline_v12["ocr_text"] != "").mean())

        rows.extend([
            {"gate": "product_changes_vs_v12", "value": len(changes12), "limit": V14_MAX_CHANGES_VS_V12, "pass": len(changes12) <= V14_MAX_CHANGES_VS_V12},
            {"gate": "product_additions_from_v12_empty", "value": len(additions12), "limit": V14_MAX_ADDITIONS_FROM_V12_EMPTY, "pass": len(additions12) <= V14_MAX_ADDITIONS_FROM_V12_EMPTY},
            {"gate": "short_ocr_additions_vs_v12", "value": len(short_additions12), "limit": 0, "pass": len(short_additions12) == 0},
            {"gate": "product_removals_vs_v12", "value": len(removals12), "limit": 0, "pass": len(removals12) == 0},
            {"gate": "unsupported_products_vs_v12", "value": len(unsupported), "limit": len(v12_unsupported), "pass": len(unsupported) <= len(v12_unsupported)},
            {"gate": "ocr_identical_to_v12", "value": int((merged12["ocr_text_v12"] != merged12["ocr_text_cand"]).sum()), "limit": 0, "pass": (merged12["ocr_text_v12"] != merged12["ocr_text_cand"]).sum() == 0},
            {"gate": "ocr_fill_not_below_v12", "value": ocr_fill, "limit": v12_ocr_fill, "pass": ocr_fill >= v12_ocr_fill},
        ])

        print("\nV14 drift vs v12:")
        print(f"  Product changes vs v12        : {len(changes12):,}")
        print(f"  Additions from v12 empty      : {len(additions12):,}")
        print(f"  Removals to empty             : {len(removals12):,}")
        print(f"  Non-empty swaps/refinements   : {len(swaps12):,}")
        print(f"  Short-OCR additions vs v12    : {len(short_additions12):,}")
        print(f"  Product fill candidate/v12    : {product_fill:.2%} / {(baseline_v12['product_name'] != '').mean():.2%}")
        print(f"  OCR changes vs v12            : {(merged12['ocr_text_v12'] != merged12['ocr_text_cand']).sum():,}")

        if len(changes12):
            print("\nProduct changes vs v12 to inspect before any submission:")
            display(changes12[["image_id", "product_name_v12", "product_name_cand", "ocr_text_cand"]].head(80))

    if prior_v9 is not None:
        merged_v9 = prior_v9.merge(candidate, on="image_id", suffixes=("_v9", "_cand"))
        print("\nDistance from v9:")
        print(f"  Product changes vs v9: {(merged_v9['product_name_v9'] != merged_v9['product_name_cand']).sum():,}")
        print(f"  OCR changes vs v9    : {(merged_v9['ocr_text_v9'] != merged_v9['ocr_text_cand']).sum():,}")

    gate_df = pd.DataFrame(rows)
    print("\nPrivate-safe submission gates:")
    display(gate_df)
    if gate_df["pass"].all():
        print("PASS: Candidate satisfies the v14 risk gates. It is eligible for offline review, then one cautious Kaggle attempt.")
    else:
        print("FAIL: Do not spend a Kaggle attempt on this candidate; keep v12/v8 as final-safe.")
    return gate_df


bad_products = audit_product_evidence(sub)
private_safe_gate_df = private_safe_risk_audit(sub)


## Cell 7 — Local Evaluation (Composite Score)

Score predictions with a **standalone inline** metric (same formula as Kaggle).

**Local dev:** uses `smce_dataset/solution.csv` (BTC only — never upload to participants).  
**Compare:** rules-only vs train-enhanced product model on the same OCR output.

In [ ]:
import json


def _inline_composite_score(solution: pd.DataFrame, submission: pd.DataFrame, row_id_column_name: str) -> float:
    """Fallback metric identical to the competition formula."""

    def _clean(val) -> str:
        return "" if pd.isna(val) else str(val).strip()

    required_cols = {"ocr_text", "product_name"}
    if row_id_column_name not in solution.columns or row_id_column_name not in submission.columns:
        raise ValueError(f"Missing id column: {row_id_column_name}")
    if not required_cols.issubset(solution.columns) or not required_cols.issubset(submission.columns):
        raise ValueError("Both solution and submission must contain ocr_text and product_name")

    sub_ids = submission[row_id_column_name]
    sol_ids = solution[row_id_column_name]
    if sub_ids.duplicated().any():
        raise ValueError("Duplicate image_id in submission")
    if set(sub_ids) != set(sol_ids):
        missing = len(set(sol_ids) - set(sub_ids))
        extra = len(set(sub_ids) - set(sol_ids))
        raise ValueError(f"Submission IDs must match solution exactly (missing {missing}, extra {extra})")

    merged = solution.merge(submission, on=row_id_column_name, suffixes=("_gt", "_pred"), how="inner")
    if merged.empty:
        raise ValueError("No matching rows after merge")

    def token_f1(gt, pred) -> float:
        gt = _clean(gt)
        pred = _clean(pred)
        if not gt and not pred:
            return 1.0
        gt_tokens = set(gt.lower().split())
        pred_tokens = set(pred.lower().split())
        if not gt_tokens or not pred_tokens:
            return 0.0
        common = gt_tokens & pred_tokens
        precision = len(common) / len(pred_tokens)
        recall = len(common) / len(gt_tokens)
        if precision + recall == 0:
            return 0.0
        return 2 * precision * recall / (precision + recall)

    def cer(gt, pred) -> float:
        gt = _clean(gt)
        pred = _clean(pred)
        if len(gt) == 0:
            return 0.0 if len(pred) == 0 else 1.0
        m, n = len(gt), len(pred)
        dp = list(range(n + 1))
        for i in range(1, m + 1):
            prev, dp[0] = dp[0], i
            for j in range(1, n + 1):
                temp = dp[j]
                dp[j] = prev if gt[i - 1] == pred[j - 1] else 1 + min(prev, dp[j], dp[j - 1])
                prev = temp
        return min(dp[n] / len(gt), 1.0)

    product_f1 = merged.apply(lambda r: token_f1(r["product_name_gt"], r["product_name_pred"]), axis=1).mean()
    avg_cer = merged.apply(lambda r: cer(r["ocr_text_gt"], r["ocr_text_pred"]), axis=1).mean()
    return round(float(0.6 * product_f1 + 0.4 * (1.0 - avg_cer)), 4)


# Standalone metric: do not load any external notebook/python file.
score_fn = _inline_composite_score
print("Using standalone inline composite score")

solution_paths = [
    Path("smce_dataset/solution.csv"),
    Path("/kaggle/input/smce-solution/solution.csv"),
]
solution_df = None
for p in solution_paths:
    if p.exists():
        solution_df = pd.read_csv(p, keep_default_na=False)
        print(f"Loaded solution: {p} ({len(solution_df):,} rows)")
        break

if solution_df is None:
    print("solution.csv not found — skip local scoring (OK on Kaggle participant side)")
elif not results:
    print("No OCR predictions yet — running oracle ablation on GT ocr_text (product head only)\n")
    gt = solution_df[["image_id", "ocr_text", "product_name"]]
    for c in ["ocr_text", "product_name"]:
        gt[c] = gt[c].astype(str).str.strip()
    sub_rules = gt.copy()
    sub_rules["product_name"] = sub_rules["ocr_text"].map(extract_product)
    sub_train = gt.copy()
    sub_train["product_name"] = sub_train["ocr_text"].map(predict_product)
    print(f"  Rules-only product:     {score_fn(gt, sub_rules, 'image_id'):.4f}")
    print(f"  Train-enhanced product: {score_fn(gt, sub_train, 'image_id'):.4f}")
    print("\nRun Cell 5 for full pipeline score (OCR + product model)")
else:
    pred_df = pd.DataFrame(results)[["image_id", "ocr_text", "product_name"]]
    pred_rules = pred_df.copy()
    pred_rules["product_name"] = pred_rules["ocr_text"].map(extract_product)

    gt = solution_df[["image_id", "ocr_text", "product_name"]]
    s_full = score_fn(gt, pred_df, "image_id")
    s_rules = score_fn(gt, pred_rules, "image_id")

    print("\nComposite score (full pipeline — OCR + train product model):")
    print(f"  Score: {s_full:.4f}")

    print("\nAblation (same OCR, rules-only product):")
    print(f"  Score: {s_rules:.4f}")

    if solution_df is not None and "Usage" in solution_df.columns:
        for split in ["Public", "Private"]:
            ids = set(solution_df.loc[solution_df["Usage"] == split, "image_id"])
            if not ids:
                continue
            gt_s = gt[gt["image_id"].isin(ids)]
            pred_s = pred_df[pred_df["image_id"].isin(ids)]
            print(f"  {split}: {score_fn(gt_s, pred_s, 'image_id'):.4f}")

    sample_empty = pd.read_csv(SAMPLE_CSV, keep_default_na=False)
    sample_empty["ocr_text"] = ""
    sample_empty["product_name"] = ""
    print(f"\nReference — empty sample_submission: {score_fn(gt, sample_empty, 'image_id'):.4f}")

## Cell 8 - Next Steps

V16 is a larger candidate than V14/V15:

- Use the VietOCR v1 bundle, not v2.
- Run once on Kaggle and download both `submission.csv` and the checkpoint CSV.
- Do not submit blindly. Compare against `submission_v14.csv` and the V15 guard replay first.
- Hard runtime rule: if the Kaggle run projects above 2 hours, stop it and fall back to V14/V15.
- The main risk is beam search changing OCR enough to alter product predictions. The risk audit below is required before spending a submission attempt.
